# 01 · 数据获取与数据溯源 (Data acquisition & provenance)

**学习目标**

1. 了解本项目使用的官方数据源，以及哪些可以自动下载、哪些必须手动下载。
2. 理解一个 "安全" 的下载函数需要处理哪些失败情况。
3. 理解数据溯源 (provenance)：URL、下载时间、SHA-256 校验和、数据版本 (vintage)。
4. 学会读取 EIA、FRED、Dallas Fed、World Bank、RBNZ 的原始文件格式。

> 本 notebook 不会发起网络请求。下载请在终端运行 `uv run python scripts/data_download.py`。
> 如果原始文件不存在（例如在 CI 中），相关单元格会跳过并说明原因。

In [1]:
import json

import numpy as np
import pandas as pd

from economic_lab.data import readers
from economic_lab.data.provenance import load_manifest
from economic_lab.data.sources import SOURCES
from economic_lab.utils.paths import manifest_path, raw_data_dir

pd.set_option("display.max_colwidth", 60)
RAW = raw_data_dir()

## 1. 数据源登记表

`economic_lab.data.sources.SOURCES` 是唯一的数据源登记表。`mode` 字段决定获取方式：

| mode | 含义 |
|---|---|
| `auto` | 直接文件链接，已验证允许脚本下载 |
| `auto_discover` | 文件链接每次发布都会变（World Bank Pink Sheet 的 URL 中含有哈希），从官方页面中搜索 |
| `rbnz_permission` | RBNZ 限制自动访问其网站（脚本请求会得到 HTTP 403）。**必须在浏览器中手动下载**，不能绕过 |
| `manual` | 其他需要浏览器或 API key 的来源（Stats NZ、MBIE） |

In [2]:
table = pd.DataFrame(SOURCES).T[["mode", "provider", "frequency", "filename"]]
table["present"] = [(RAW / f).exists() for f in table["filename"]]
table

,mode,provider,frequency,filename,present
brent_daily,auto,U.S. Energy Information Administration,daily,eia/brent_daily.xls,True
brent_monthly,auto,U.S. Energy Information Administration,monthly,eia/brent_monthly.xls,True
nzd_usd_daily,auto,Board of Governors of the Federal Reserve System (H.10) ...,daily,fred/DEXUSNZ.csv,True
nz_cpi_oecd,auto,OECD Main Economic Indicators via FRED (original source:...,quarterly,fred/NZLCPIALLQINMEI.csv,True
dallasfed_igrea,auto,Federal Reserve Bank of Dallas,monthly,dallasfed/igrea.xlsx,True
worldbank_pinksheet_monthly,auto_discover,World Bank Commodity Markets (Pink Sheet),monthly,worldbank/CMO-Historical-Data-Monthly.xlsx,True
rbnz_cpi,rbnz_permission,Reserve Bank of New Zealand,quarterly,rbnz/hm1.xlsx,True
rbnz_gdp,rbnz_permission,Reserve Bank of New Zealand,quarterly,rbnz/hm5.xlsx,True
rbnz_exchange_rates,rbnz_permission,Reserve Bank of New Zealand,daily,rbnz/hb1-daily.xlsx,True
rbnz_interest_rates,rbnz_permission,Reserve Bank of New Zealand,daily,rbnz/hb2-daily-close.xlsx,True


### 为什么 NZD/USD 汇率用 FRED？

RBNZ B1 是新西兰汇率的首选来源，但不能自动下载。FRED 的 `DEXUSNZ` 是美联储 H.10 发布的纽约午间买入汇率
（每 1 NZD 兑多少 USD），是官方数据的再发布，可以脚本化获取。两者在同一天的报价时点不同（纽约中午 vs 惠灵顿），
对季度均值的影响很小，但做日度分析时要注意。

### 为什么有一个 OECD CPI？

`nz_cpi_oecd` (FRED: `NZLCPIALLQINMEI`) 只是 **备用的总体 CPI**：它可以自动下载，但 OECD MEI 已停止更新
（最后观测为 2025Q1），而且没有 tradable / non-tradable 分项。手动下载 RBNZ M1 后，构建脚本会优先使用 RBNZ 数据。

## 2. 安全下载：原来的函数会在哪里失败？

原始版本：

```python
response = requests.get(url)
with open(file_path, "wb") as f:
    f.write(response.content)
```

| 失败情形 | 后果 | 新版本的处理 |
|---|---|---|
| 服务器返回 403/404 | HTML 错误页面被保存为 `hm1.xlsx` | `response.raise_for_status()` |
| 服务器返回 200 但内容是 HTML | 同上 | 检查文件头 magic bytes (`check_file_format`) |
| 无超时 | 脚本可能永远挂起 | `timeout=120` |
| 下载中断 | 留下截断文件；下次运行看到"文件已存在"而跳过 | 先写入 `.part`，校验后原子重命名 |
| 刷新数据 | 旧版本被覆盖，无法复现旧结果 | 旧文件移入 `_vintages/` 并加时间戳 |

下面演示文件格式检查：`.xlsx` 是 zip 容器，必须以 `PK\x03\x04` 开头；`.xls` 是 OLE2 文档，以 `D0 CF 11 E0` 开头。

In [3]:
import tempfile
from pathlib import Path

from economic_lab.data.downloader import check_file_format

with tempfile.TemporaryDirectory() as tmp:
    fake = Path(tmp) / "hm1.xlsx"
    fake.write_bytes(b"<!DOCTYPE html><html><body>Access denied</body></html>")
    try:
        check_file_format(fake)
    except ValueError as err:
        print("Rejected:", err)

    good = Path(tmp) / "ok.xlsx"
    pd.DataFrame({"a": [1]}).to_excel(good)
    check_file_format(good)
    print("Accepted:", good.name, good.read_bytes()[:4])

Rejected: hm1.xlsx: content is not a valid .xlsx file
Accepted: ok.xlsx b'PK\x03\x04'


## 3. 数据溯源：manifest

每次下载（或登记手动下载的文件）都会写入 `data/econ_data/manifest.json`：URL、UTC 时间、SHA-256、
文件大小和 HTTP 头 (`Last-Modified`, `ETag`)。

**为什么需要校验和？** 官方宏观数据会被修订 (revision)。同一个文件名，下个月下载的内容可能不同。
如果你的报告写着 "oil pass-through = 0.017"，别人需要知道这是用哪一版数据算出来的。
SHA-256 改变 ⇒ 数据源发布了新的 vintage；旧条目保留在 `history` 中。

In [4]:
manifest = load_manifest(manifest_path())
if manifest["files"]:
    m = pd.DataFrame(manifest["files"]).T[["method", "recorded_at_utc", "size_bytes", "sha256"]]
    m["sha256"] = m["sha256"].str[:12] + "…"
    display(m)
else:
    print("No manifest yet: run `uv run python scripts/data_download.py`.")

,method,recorded_at_utc,size_bytes,sha256
brent_daily,automatic,2026-10-12T10:23:26+00:00,452096,a7b7074cb3e1…
brent_monthly,automatic,2026-10-12T10:23:26+00:00,55296,834d47313c17…
dallasfed_igrea,automatic,2026-10-12T10:23:28+00:00,27832,edfafd9b139c…
nz_cpi_oecd,automatic,2026-10-12T10:23:27+00:00,13593,9952c8c84a86…
nzd_usd_daily,automatic,2026-10-12T10:23:27+00:00,258355,49a06beab71d…
rbnz_cpi,manual,2026-10-12T21:29:37+00:00,48280,523a1326160f…
rbnz_exchange_rates,manual,2026-10-12T21:29:37+00:00,297928,e9f9c88c7dde…
rbnz_expectations,manual,2026-10-12T21:29:37+00:00,33387,93919c296472…
rbnz_gdp,manual,2026-10-12T21:29:37+00:00,71077,9437e238eefe…
rbnz_interest_rates,manual,2026-10-12T21:29:37+00:00,432598,66f2504f0ecd…


## 4. 读取原始文件

每个数据源有自己的格式。读取器 (reader) 只负责 **忠实地解析格式**，不做经济含义上的清洗；
清洗规则集中在 `economic_lab.data.build` 中并写入元数据。下面依次查看每个原始文件。

In [5]:
def show(name, reader):
    path = RAW / SOURCES[name]["filename"]
    if not path.exists():
        print(f"[skip] {name}: {path.name} not downloaded")
        return None
    obj = reader(path)
    print(f"{name}: {type(obj).__name__}, {len(obj)} rows, "
          f"{obj.index.min():%Y-%m-%d} → {obj.index.max():%Y-%m-%d}")
    return obj

brent_d = show("brent_daily", readers.read_eia_xls)
brent_m = show("brent_monthly", readers.read_eia_xls)
fx_d = show("nzd_usd_daily", lambda p: readers.read_fred_csv(p, "nzd_usd", "USD per NZD"))
cpi_oecd = show("nz_cpi_oecd", lambda p: readers.read_fred_csv(p, "cpi", "index"))
igrea = show("dallasfed_igrea", readers.read_dallasfed_igrea)
pink = show("worldbank_pinksheet_monthly", readers.read_worldbank_pinksheet)

brent_daily: Series, 9102 rows, 1987-05-20 → 2026-10-06
brent_monthly: Series, 473 rows, 1987-05-15 → 2026-09-15
nzd_usd_daily: Series, 13965 rows, 1971-01-04 → 2026-10-02
nz_cpi_oecd: Series, 444 rows, 1914-04-01 → 2025-01-01
dallasfed_igrea: Series, 704 rows, 1968-01-01 → 2026-08-01


worldbank_pinksheet_monthly: DataFrame, 801 rows, 1960-01-01 → 2026-09-01


### 原始数据中真实存在的问题

在构建本项目时，在官方文件中发现了以下问题 —— 这正是数据校验 (validation) 必不可少的原因：

1. **OECD/FRED 新西兰 CPI 含有 `0.0` 占位值**（1914–1925 年间 27 个）。价格指数不可能为 0；
   若直接计算增长率会得到 −100% 或无穷大。
2. **EIA 月度 Brent 的日期是每月 15 日**，而 FRED 季度数据用季初日期；不统一日期约定，`join` 时会全部错位。
3. **Dallas Fed IGREA 最近几行的日期是月中**（如 `2026-07-26`）而不是月初。
4. **EIA 日度 Brent 在 1994–2002 年很稀疏**：例如 1995Q1 只有 26 个交易日（应有 65 个）。
   大多数缺口短于一周，所以单看 "最长缺口" 发现不了，需要检查覆盖率。

In [6]:
if cpi_oecd is not None:
    zeros = cpi_oecd[cpi_oecd <= 0]
    print(f"Zero placeholders in OECD CPI: {len(zeros)}, from {zeros.index.min():%Y} to {zeros.index.max():%Y}")
if brent_m is not None:
    print("EIA monthly day-of-month values:", sorted(brent_m.index.day.unique()))
if igrea is not None:
    print("IGREA (after reader normalisation) day-of-month values:", sorted(igrea.index.day.unique()))
if brent_d is not None:
    per_year = brent_d.groupby(brent_d.index.year).size()
    bdays = pd.Series({y: len(pd.bdate_range(f"{y}-01-01", f"{y}-12-31")) for y in per_year.index})
    coverage = (per_year / bdays).loc[1990:2008]
    print("EIA daily Brent coverage by year (share of business days):")
    print(coverage.round(2).to_string())

Zero placeholders in OECD CPI: 27, from 1914 to 1925
EIA monthly day-of-month values: [15]
IGREA (after reader normalisation) day-of-month values: [1]
EIA daily Brent coverage by year (share of business days):
date
1990    0.97
1991    0.96
1992    0.94
1993    0.82
1994    0.53
1995    0.49
1996    0.70
1997    0.71
1998    0.72
1999    0.66
2000    0.78
2001    0.74
2002    0.74
2003    0.83
2004    0.88
2005    0.86
2006    0.88
2007    0.95
2008    0.97


## 5. RBNZ 手动下载流程

1. 在浏览器中打开 <https://www.rbnz.govt.nz/statistics/series/data-file-index-page>，找到 M1 (Prices) 等表格，下载 Excel 文件。
2. 保持 RBNZ 原文件名，保存到 `data/econ_data/rbnz/hm1.xlsx`（文件名见 `SOURCES`）。
3. 运行 `uv run python scripts/data_download.py`：它会把手动文件登记到 manifest（`method: manual`）。
4. 运行 `uv run python scripts/inspect_rbnz.py data/econ_data/rbnz/hm1.xlsx`，查看每个 series id 的描述。
5. 需要新变量时，把 series id 加入 `configs/variables.yaml` 的 `rbnz_series`（CPI、GDP、失业率、OCR 等已配置好）。
6. 运行 `uv run python scripts/build_datasets.py`。

RBNZ 文件的标题行数随时间变化，所以 `read_rbnz_xlsx` 不硬编码行号，而是搜索第一列为 `Series Id` 的那一行。
下面用一个按相同结构构造的小文件演示（**这是合成的示例文件，不是 RBNZ 数据**）：

In [7]:
rows = [
    ["Series", "CPI (all groups)", "Tradables CPI"],
    ["Unit", "Index", "Index"],
    ["Series Id", "SERIES.A", "SERIES.B"],
    [pd.Timestamp("2024-03-31"), 1000.0, 1000.0],
    [pd.Timestamp("2024-06-30"), 1010.0, 1004.0],
]
with tempfile.TemporaryDirectory() as tmp:
    f = Path(tmp) / "demo.xlsx"
    pd.DataFrame(rows).to_excel(f, sheet_name="Data", header=False, index=False)
    data, meta = readers.read_rbnz_xlsx(f)
display(meta)
display(data)

,Series,Unit
series_id,,
SERIES.A,CPI (all groups),Index
SERIES.B,Tradables CPI,Index


,SERIES.A,SERIES.B
date,,
2024-03-31,1000,1000
2024-06-30,1010,1004


## 练习

1. **Easy** — 在 `SOURCES` 中找到 World Bank Pink Sheet 的条目。为什么它的 `url` 是 `None`？`discover_pattern` 匹配的是什么？
2. **Intermediate** — 写一个函数 `changed_sources(manifest)`，列出 `history` 非空（即至少有过一次数据修订）的数据源。
   用 `tempfile` 构造一个两次登记不同内容的 manifest 来测试它。
3. **Advanced** — EIA 日度 Brent 在 1990 年代稀疏。如果你一定要用日度数据构建季度均值，有哪些选择？
   （提示：仅用有观测的日子求平均、插值、或换数据源。）分别会引入什么偏差？

## 参考答案

1. Pink Sheet 的文件 URL 包含一个每月变化的哈希（例如 `.../74e8be41...-0050012026/related/CMO-Historical-Data-Monthly.xlsx`），
   写死会在下个月失效。`discover_pattern` 是正则表达式，在官方 Commodity Markets 页面的 HTML 中搜索以
   `CMO-Historical-Data-Monthly.xlsx` 结尾的链接。
2. 示例：
   ```python
   def changed_sources(manifest: dict) -> list[str]:
       return [k for k, v in manifest["files"].items() if v.get("history")]

   from economic_lab.data.provenance import record_file
   with tempfile.TemporaryDirectory() as tmp:
       f = Path(tmp) / "x.csv"; mpath = Path(tmp) / "m.json"
       f.write_text("v1"); record_file(mpath, "x", f, url=None, method="manual")
       f.write_text("v2"); record_file(mpath, "x", f, url=None, method="manual")
       print(changed_sources(load_manifest(mpath)))   # ['x']
   ```
3. (a) 仅用有观测的日子求平均：如果缺失与价格无关（missing completely at random），估计无偏但更嘈杂；
   如果缺失集中在季度某一段（例如季末），会偏向季度的那一部分，在油价趋势明显时产生偏差。
   (b) 线性插值：人为平滑价格，低估波动率，并让 "日度" 数据包含未来信息（插值用到了缺口之后的价格），
   这在预测问题中就是 look-ahead bias。
   (c) 换用 EIA 月度平均（本项目的做法）：EIA 用完整数据计算，覆盖率无问题；代价是 NZD 换算只能在月度层面完成，
   即 "比值的均值" 变成 "均值的比值"。`oil_nzd_from_daily` 列让你量化这个差异（见 notebook 02）。